# Day 17 — Feature engineering: encoding, scaling and derived features

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction: why feature engineering matters

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-017/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd

# The lesson's five students: raw features
df = pd.DataFrame({"study_hours": [2, 4, 6, 3, 8], "attendance": [55, 70, 90, 65, 95],
                   "prev_score": [40, 55, 85, 50, 92], "assignments_done": [3, 5, 9, 4, 10]})
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done
           2          55          40                 3
           4          70          55                 5
           6          90          85                 9
           3          65          50                 4
           8          95          92                10


In [ ]:
# Feature creation and binning
df["weekly_study_hours"] = df["study_hours"] * 7
df["attendance_category"] = pd.cut(df["attendance"], bins=[0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["score_category"] = pd.cut(df["prev_score"], bins=[0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["assignment_completion"] = df["assignments_done"] / 10          # completion ratio, assuming 10 assignments were set
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done  weekly_study_hours attendance_category score_category  assignment_completion
           2          55          40                 3                  14                 Low            Low                    0.3
           4          70          55                 5                  28              Medium         Medium                    0.5
           6          90          85                 9                  42                High           High                    0.9
           3          65          50                 4                  21              Medium            Low                    0.4
           8          95          92                10                  56                High           High                    1.0


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Model A vs Model B on our 127 training students: does engineering features help? (pass/fail, 5-fold CV)
data = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(data, test_size=0.15, random_state=42, stratify=data["Passed"])
y = train_val["Passed"].astype(int)
X = train_val[["StudyHours", "Attendance", "PreviousScore"]].copy()
X["WeeklyStudyHours"] = X["StudyHours"] * 7
X["AttendanceCategory"] = pd.cut(X["Attendance"], [0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
X["ScoreCategory"] = pd.cut(X["PreviousScore"], [0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def model(numeric, categorical=()):
    prep = ColumnTransformer([("num", StandardScaler(), list(numeric))] +
                             ([("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical))] if categorical else []))
    return make_pipeline(prep, LogisticRegression(max_iter=1000))
raw = ["StudyHours", "Attendance", "PreviousScore"]
for name, num, cat in [("A: raw features", raw, ()),
                       ("A + weekly study hours", raw + ["WeeklyStudyHours"], ()),
                       ("Categories only (binned)", ["StudyHours"], ["AttendanceCategory", "ScoreCategory"]),
                       ("Raw + categories", raw, ["AttendanceCategory", "ScoreCategory"])]:
    s = cross_val_score(model(num, cat), X, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f} +/- {s.std():.3f}")

A: raw features            CV accuracy 0.796 +/- 0.062
A + weekly study hours     CV accuracy 0.788 +/- 0.073
Categories only (binned)   CV accuracy 0.781 +/- 0.089


Raw + categories           CV accuracy 0.796 +/- 0.062


In [ ]:
# Data leakage: a feature that would not exist at prediction time. ExamScore decides pass/fail, so a "feature" built
# from it looks perfect in testing and is useless in practice (we never know it before the exam)
X_leak = X[raw].assign(ScoreChange=train_val["ExamScore"] - train_val["PreviousScore"])
for name, cols, frame in [("Raw features", raw, X), ("Raw + ScoreChange (leaks)", raw + ["ScoreChange"], X_leak)]:
    s = cross_val_score(model(cols), frame, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f}")

Raw features               CV accuracy 0.796
Raw + ScoreChange (leaks)  CV accuracy 0.937


## 2. Categorical encoding: label, one-hot and ordinal

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encoding: each category gets a number, in alphabetical order
data = ["Male", "Female", "Female", "Male"]
encoder = LabelEncoder()
encoded = encoder.fit_transform(data)
print(encoded)
print(encoder.classes_)

# A target label: alphabetical order makes Fail 0 and Pass 1
results = ["Pass", "Fail", "Pass", "Fail", "Pass"]
y_encoder = LabelEncoder()
print(y_encoder.fit_transform(results), y_encoder.classes_)

[1 0 0 1]
['Female' 'Male']
[1 0 1 0 1] ['Fail' 'Pass']


In [ ]:
# One-hot encoding with pandas: one 0/1 column per city
df = pd.DataFrame({"student": ["A", "B", "C", "D"],
                   "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad"]})
encoded_df = pd.get_dummies(df, columns=["city"], dtype=int)
print(encoded_df.to_string(index=False))      # columns come out in alphabetical order

student  city_Delhi  city_Hyderabad  city_Mumbai
      A           0               1            0
      B           0               0            1
      C           1               0            0
      D           0               1            0


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Ordinal encoding: we state the order, Low < Medium < High
levels = [["Low"], ["Medium"], ["High"], ["Medium"]]
encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
print(encoder.fit_transform(levels).ravel())

# Without the order, categories are sorted alphabetically: High 0, Low 1, Medium 2 (wrong order)
print(OrdinalEncoder().fit_transform(levels).ravel())

[0. 1. 2. 1.]
[1. 2. 0. 2.]


In [ ]:
# Our dataset: Class is a nominal category (no order), with 5 students recorded as "Unknown"
train_x = train_val[["StudyHours", "Attendance", "PreviousScore", "Class"]].replace({"Class": {"Unknown": np.nan}})
print(train_x["Class"].value_counts(dropna=False).to_string())
print("Pass rate by class:", y.groupby(train_x["Class"]).mean().round(2).to_dict())

# Fit the encoder on training data only; a class never seen in training ("D") becomes all zeros
onehot = OneHotEncoder(handle_unknown="ignore").fit(train_x[["Class"]].dropna())
print(onehot.categories_[0])
print(onehot.transform(pd.DataFrame({"Class": ["B", "D"]})).toarray())
try:
    OneHotEncoder().fit(train_x[["Class"]].dropna()).transform(pd.DataFrame({"Class": ["D"]}))
except ValueError as error:
    print("Without handle_unknown='ignore':", str(error)[:60])

Class
A      43
B      42
C      37
NaN     5
Pass rate by class: {'A': 0.72, 'B': 0.74, 'C': 0.68}
['A' 'B' 'C']
[[0. 1. 0.]
 [0. 0. 0.]]
Without handle_unknown='ignore': Found unknown categories ['D'] in column 0 during transform


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# A production-style pipeline: impute and scale numbers, impute and one-hot encode categories, then a model.
# Every step is fitted on the training folds only, so nothing leaks from the fold being scored
numeric_features = ["StudyHours", "Attendance", "PreviousScore"]
categorical_features = ["Class"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])
pipe = Pipeline([("preprocessor", preprocessor),
                 ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))])
with_class = cross_val_score(pipe, train_x, y, cv=cv)
without_class = cross_val_score(pipe.set_params(preprocessor__categorical="drop"), train_x, y, cv=cv)
print(f"With Class    CV accuracy {with_class.mean():.3f}")
print(f"Without Class CV accuracy {without_class.mean():.3f}")

# Fit once on all 127 training students and score two new students, one from a class never seen
pipe.set_params(preprocessor__categorical=categorical_pipeline).fit(train_x, y)
new = pd.DataFrame({"StudyHours": [5, 5], "Attendance": [85, 85], "PreviousScore": [70, 70], "Class": ["B", "D"]})
print("P(pass):", pipe.predict_proba(new)[:, 1].round(2))

With Class    CV accuracy 0.780
Without Class CV accuracy 0.804
P(pass): [1.   0.99]


## 3. Feature scaling: standardization, normalization and robust scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Three scalers on five students: each column is scaled on its own
df = pd.DataFrame({"study_hours": [2, 4, 6, 8, 10],
                   "attendance": [50, 60, 75, 85, 95],
                   "family_income": [20000, 30000, 45000, 60000, 90000]})
print(df.to_string(index=False))
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    scaled_df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    print(f"\n{type(scaler).__name__}")
    print(scaled_df.round(3).to_string(index=False))

 study_hours  attendance  family_income
           2          50          20000
           4          60          30000
           6          75          45000
           8          85          60000
          10          95          90000

StandardScaler
 study_hours  attendance  family_income
      -1.414      -1.410         -1.180
      -0.707      -0.797         -0.773
       0.000       0.123         -0.163
       0.707       0.736          0.448
       1.414       1.349          1.668

MinMaxScaler
 study_hours  attendance  family_income
        0.00       0.000          0.000
        0.25       0.222          0.143
        0.50       0.556          0.357
        0.75       0.778          0.571
        1.00       1.000          1.000

RobustScaler
 study_hours  attendance  family_income
        -1.0        -1.0         -0.833
        -0.5        -0.6         -0.500
         0.0         0.0          0.000
         0.5         0.4          0.500
         1.0         0.8          1.

In [ ]:
# The impact of an outlier: 1000 among 10, 20, 30, 40
values = np.array([[10], [20], [30], [40], [1000]])
print("mean", values.mean(), "| std", values.std().round(1), "| median", np.median(values),
      "| IQR", np.percentile(values, 75) - np.percentile(values, 25))
for name, scaler in {"Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler()}.items():
    print(f"{name:<9}", scaler.fit_transform(values).ravel().round(2))

mean 220.0 | std 390.1 | median 30.0 | IQR 20.0
Standard  [-0.54 -0.51 -0.49 -0.46  2.  ]
MinMax    [0.   0.01 0.02 0.03 1.  ]
Robust    [-1.  -0.5  0.   0.5 48.5]


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Does scaling matter on our data? Same 127 training students, same 5 folds, with and without StandardScaler
features = ["StudyHours", "Attendance", "PreviousScore"]
print(train_val[features].agg(["min", "max"]).to_string())
for name, clf in [("KNN (k=5)", KNeighborsClassifier()),
                  ("Logistic regression", LogisticRegression(max_iter=1000)),
                  ("Decision tree", DecisionTreeClassifier(max_depth=3, random_state=42))]:
    raw_cv = cross_val_score(clf, train_val[features], y, cv=cv).mean()
    scaled_cv = cross_val_score(make_pipeline(StandardScaler(), clf), train_val[features], y, cv=cv).mean()
    print(f"{name:<20} unscaled {raw_cv:.3f} | scaled {scaled_cv:.3f}")

# Why does the tree change at all? Scaling keeps the order, so the splits are the same; look at the students it changed
from sklearn.model_selection import cross_val_predict
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
raw_pred = cross_val_predict(tree, train_val[features], y, cv=cv)
scaled_pred = cross_val_predict(make_pipeline(StandardScaler(), tree), train_val[features], y, cv=cv)
print("Tree predictions that differ:", (raw_pred != scaled_pred).sum(), "of", len(y))
print(train_val[features][raw_pred != scaled_pred].to_string(index=False))

     StudyHours  Attendance  PreviousScore
min         0.5        55.0             27
max         7.0       100.0             99


KNN (k=5)            unscaled 0.772 | scaled 0.795
Logistic regression  unscaled 0.796 | scaled 0.796


Decision tree        unscaled 0.717 | scaled 0.733


Tree predictions that differ: 2 of 127
 StudyHours  Attendance  PreviousScore
        4.5        86.0             44
        4.5        79.0             48


In [ ]:
# Leakage: fitting the scaler on all 127 students before cross-validation lets each fold's scoring students shape it
leaky_X = StandardScaler().fit_transform(train_val[features])
leaky = cross_val_score(KNeighborsClassifier(), leaky_X, y, cv=cv).mean()
model = Pipeline([("scaler", StandardScaler()), ("classifier", KNeighborsClassifier())])
correct = cross_val_score(model, train_val[features], y, cv=cv).mean()
print(f"Scaler fitted before splitting: {leaky:.3f} | inside the pipeline: {correct:.3f}")

# The pipeline learns the scaler's mean and spread from the training data it is fitted on
model.fit(train_val[features], y)
print("Learned means:", model.named_steps["scaler"].mean_.round(2))
print("Learned stds: ", model.named_steps["scaler"].scale_.round(2))

Scaler fitted before splitting: 0.771 | inside the pipeline: 0.795
Learned means: [ 3.61 79.13 62.39]
Learned stds:  [ 1.37 10.   15.11]


## 4. Derived features: creating new features from existing data

In [ ]:
# Derived features: four students
df = pd.DataFrame({"student_id": [101, 102, 103, 104], "study_hours": [2, 4, 6, 3],
                   "attendance": [55, 70, 90, 65], "assignments_done": [3, 5, 9, 4],
                   "total_assignments": [5, 5, 10, 5], "previous_score": [40, 55, 85, 50]})

# Mathematical features
df["weekly_study_hours"] = df["study_hours"] * 7
df["score_difference"] = df["previous_score"] - 50          # above (+) or below (-) a pass mark of 50
df["study_hours_squared"] = df["study_hours"] ** 2

# A ratio: replace a zero total with missing, so a division by zero gives NaN instead of an error or infinity
df["assignment_completion_pct"] = df["assignments_done"] / df["total_assignments"].replace(0, np.nan) * 100
print(df[["student_id", "study_hours", "weekly_study_hours", "score_difference", "study_hours_squared",
          "assignment_completion_pct"]].to_string(index=False))

 student_id  study_hours  weekly_study_hours  score_difference  study_hours_squared  assignment_completion_pct
        101            2                  14               -10                    4                       60.0
        102            4                  28                 5                   16                      100.0
        103            6                  42                35                   36                       90.0
        104            3                  21                 0                    9                       80.0


In [ ]:
# Date and time features
df_dates = pd.DataFrame({"admission_date": pd.to_datetime(["2026-06-10", "2026-07-15", "2026-08-20", "2026-09-05"])})
df_dates["admission_month"] = df_dates["admission_date"].dt.month
df_dates["admission_weekday"] = df_dates["admission_date"].dt.day_of_week     # Monday 0 ... Sunday 6
df_dates["day"] = df_dates["admission_date"].dt.day_name()
df_dates["admission_quarter"] = df_dates["admission_date"].dt.quarter
df_dates["days_since"] = (pd.Timestamp("2026-10-01") - df_dates["admission_date"]).dt.days   # at prediction date
print(df_dates.rename(columns=lambda c: c.replace("admission_", "")).to_string(index=False))

      date  month  weekday       day  quarter  days_since
2026-06-10      6        2 Wednesday        2         113
2026-07-15      7        2 Wednesday        3          78
2026-08-20      8        3  Thursday        3          42
2026-09-05      9        5  Saturday        3          26


In [ ]:
# Aggregation features over three earlier exams, and a domain-specific engagement score
exams = ["exam_1", "exam_2", "exam_3"]
df["exam_1"], df["exam_2"], df["exam_3"] = [60, 70, 80, 55], [65, 75, 85, 60], [70, 80, 90, 65]
df["average_exam_score"] = df[exams].mean(axis=1)
df["highest_exam_score"] = df[exams].max(axis=1)
df["lowest_exam_score"] = df[exams].min(axis=1)

# Engagement (0 to 100): weights chosen from domain knowledge; study hours as a share of an 8-hour day
df["study_pct"] = (df["study_hours"] / 8 * 100).clip(upper=100)
df["engagement_score"] = 0.5 * df["attendance"] + 0.3 * df["assignment_completion_pct"] + 0.2 * df["study_pct"]
short = {"average_exam_score": "average", "highest_exam_score": "highest", "lowest_exam_score": "lowest",
         "engagement_score": "engagement"}
print(df[["student_id"] + list(short)].rename(columns=short).to_string(index=False))

 student_id  average  highest  lowest  engagement
        101     65.0       70      60        50.5
        102     75.0       80      70        75.0
        103     85.0       90      80        87.0
        104     60.0       65      55        64.0


In [ ]:
# Do derived features help on our students? Logistic regression, 5-fold CV on the 127 training students
base = train_val[["StudyHours", "Attendance", "PreviousScore"]].copy()
derived = base.assign(StudyHoursSquared=base["StudyHours"] ** 2,
                      StudyXAttendance=base["StudyHours"] * base["Attendance"],
                      Engagement=0.5 * base["Attendance"] + 0.5 * (base["StudyHours"] / 8 * 100))
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, cols in [("Raw features", ["StudyHours", "Attendance", "PreviousScore"]),
                   ("+ study hours squared", ["StudyHours", "Attendance", "PreviousScore", "StudyHoursSquared"]),
                   ("+ study x attendance", ["StudyHours", "Attendance", "PreviousScore", "StudyXAttendance"]),
                   ("Engagement + previous", ["Engagement", "PreviousScore"])]:
    print(f"{name:<24} CV accuracy {cross_val_score(lr, derived[cols], y, cv=cv).mean():.3f}")

Raw features             CV accuracy 0.796


+ study hours squared    CV accuracy 0.796
+ study x attendance     CV accuracy 0.796


Engagement + previous    CV accuracy 0.780


In [ ]:
# Leakage in an aggregation: "pass rate of students with the same attendance" (a group average of the target).
# Computed on all 127 students, each scored student's own result is inside the average it is given
def group_mean(fit_rows, apply_rows):
    means = train_val.loc[fit_rows].groupby("Attendance")["Passed"].mean()
    return train_val.loc[apply_rows, "Attendance"].map(means).fillna(train_val.loc[fit_rows, "Passed"].mean())

leaky_feature = group_mean(train_val.index, train_val.index)
leaky, correct = [], []
for fit_idx, score_idx in cv.split(base, y):
    fit_rows, score_rows = train_val.index[fit_idx], train_val.index[score_idx]
    for scores, fit_feature, score_feature in [
            (leaky, leaky_feature[fit_rows], leaky_feature[score_rows]),                                  # all 127
            (correct, group_mean(fit_rows, fit_rows), group_mean(fit_rows, score_rows))]:                 # training fold only
        X_fit = base.loc[fit_rows].assign(GroupMean=fit_feature.values)
        X_score = base.loc[score_rows].assign(GroupMean=score_feature.values)
        scores.append(lr.fit(X_fit, y.loc[fit_rows]).score(X_score, y.loc[score_rows]))
print("Students alone in their attendance group:", (train_val["Attendance"].value_counts() == 1).sum())
print(f"Group average from all students:     CV accuracy {np.mean(leaky):.3f}")
print(f"Group average from the training fold: CV accuracy {np.mean(correct):.3f}")

Students alone in their attendance group: 7
Group average from all students:     CV accuracy 0.851
Group average from the training fold: CV accuracy 0.725


## 5. Feature selection: choosing the most useful features

In [ ]:
# Candidate features for the 127 training students: four real ones plus three that shouldn't be there
rng = np.random.default_rng(42)
cand = train_val[["StudyHours", "Attendance", "PreviousScore", "Age"]].copy()
cand["StudentNumber"] = range(1, len(cand) + 1)                 # an identifier
cand["RandomReference"] = rng.integers(1000, 9999, len(cand))   # noise
cand["ScoreCopy"] = cand["PreviousScore"]                       # a duplicate column
corr = cand.assign(Passed=y).corr()
short = ["Study", "Attend", "Prev", "Age", "ID", "Random", "Copy", "Passed"]      # short labels to fit the screen
print(corr.set_axis(short, axis=0).set_axis(short, axis=1).round(2).to_string())

        Study  Attend  Prev   Age    ID  Random  Copy  Passed
Study    1.00    0.28  0.30  0.12  0.01   -0.03  0.30    0.50
Attend   0.28    1.00  0.08 -0.01 -0.01    0.00  0.08    0.31
Prev     0.30    0.08  1.00  0.03 -0.03   -0.17  1.00    0.38
Age      0.12   -0.01  0.03  1.00  0.12   -0.04  0.03    0.03
ID       0.01   -0.01 -0.03  0.12  1.00   -0.04 -0.03   -0.01
Random  -0.03    0.00 -0.17 -0.04 -0.04    1.00 -0.17    0.03
Copy     0.30    0.08  1.00  0.03 -0.03   -0.17  1.00    0.38
Passed   0.50    0.31  0.38  0.03 -0.01    0.03  0.38    1.00


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Filter: ANOVA F-test, keep the 3 highest-scoring features
selector = SelectKBest(score_func=f_classif, k=3).fit(cand, y)
print("Selected:", list(cand.columns[selector.get_support()]))
print(pd.Series(selector.scores_, index=cand.columns).round(1).sort_values(ascending=False).to_string())

Selected: ['StudyHours', 'PreviousScore', 'ScoreCopy']
StudyHours         40.9
PreviousScore      20.7
ScoreCopy          20.7
Attendance         13.1
Age                 0.1
RandomReference     0.1
StudentNumber       0.0


In [ ]:
from sklearn.feature_selection import RFE

# Wrapper: recursive feature elimination with a random forest, down to 3 features
rfe = RFE(estimator=RandomForestClassifier(n_estimators=100, random_state=42), n_features_to_select=3).fit(cand, y)
print("Selected:", list(cand.columns[rfe.support_]))
print(pd.Series(rfe.ranking_, index=cand.columns, name="rank").sort_values().to_string())   # 1 = kept

Selected: ['StudyHours', 'Attendance', 'PreviousScore']
StudyHours         1
Attendance         1
PreviousScore      1
ScoreCopy          2
StudentNumber      3
RandomReference    4
Age                5


In [ ]:
# Embedded: L1-regularized logistic regression sets some coefficients to exactly zero
# (scikit-learn 1.8+ warns that penalty= is deprecated and prefers l1_ratio=1; penalty="l1" works on every version)
l1 = Pipeline([("scaler", StandardScaler()),
               ("classifier", LogisticRegression(penalty="l1", solver="liblinear", C=0.1, random_state=42))]).fit(cand, y)
print(pd.DataFrame({"Feature": cand.columns, "Coefficient": l1.named_steps["classifier"].coef_[0].round(3)}).to_string(index=False))

        Feature  Coefficient
     StudyHours        0.588
     Attendance        0.102
  PreviousScore        0.243
            Age        0.000
  StudentNumber        0.000
RandomReference        0.000
      ScoreCopy        0.000


/tmp/claude-0/-home-user/371a76c3-0252-5c27-a309-cbedb0f8aa45/scratchpad/localvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/tmp/claude-0/-home-user/371a76c3-0252-5c27-a309-cbedb0f8aa45/scratchpad/localvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [ ]:
# Before and after selection: logistic regression, the same 5 folds; selection refitted inside each fold
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, model, cols in [
        ("All 7 candidate features", lr, list(cand.columns)),
        ("SelectKBest k=3 in the pipeline", make_pipeline(StandardScaler(), SelectKBest(f_classif, k=3),
                                                          LogisticRegression(max_iter=1000)), list(cand.columns)),
        ("Hand-picked: study, attendance, previous", lr, ["StudyHours", "Attendance", "PreviousScore"])]:
    print(f"{name:<42} CV accuracy {cross_val_score(model, cand[cols], y, cv=cv).mean():.3f}")

# Stability: which features does SelectKBest keep in each training fold?
for k, (fit_idx, _) in enumerate(cv.split(cand, y), 1):
    kept = cand.columns[SelectKBest(f_classif, k=3).fit(cand.iloc[fit_idx], y.iloc[fit_idx]).get_support()]
    print(f"Fold {k}:", list(kept))

All 7 candidate features                   CV accuracy 0.772
SelectKBest k=3 in the pipeline            CV accuracy 0.804
Hand-picked: study, attendance, previous   CV accuracy 0.796
Fold 1: ['StudyHours', 'PreviousScore', 'ScoreCopy']
Fold 2: ['StudyHours', 'PreviousScore', 'ScoreCopy']
Fold 3: ['StudyHours', 'Attendance', 'ScoreCopy']
Fold 4: ['StudyHours', 'PreviousScore', 'ScoreCopy']
Fold 5: ['StudyHours', 'PreviousScore', 'ScoreCopy']


In [ ]:
# Why selection must happen inside cross-validation: 500 columns of pure noise, which can't predict anything.
# Choosing the best 10 using all students first, then cross-validating, finds "signal" that isn't there
noise = pd.DataFrame(rng.normal(size=(len(y), 500)), index=y.index)
print(f"Always predicting pass: accuracy {y.mean():.3f}")
best10 = noise.columns[SelectKBest(f_classif, k=10).fit(noise, y).get_support()]
outside = cross_val_score(lr, noise[best10], y, cv=cv).mean()
inside = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), StandardScaler(),
                                       LogisticRegression(max_iter=1000)), noise, y, cv=cv).mean()
print(f"Selected on all students, then CV: {outside:.3f}")
print(f"Selected inside each fold:         {inside:.3f}")

Always predicting pass: accuracy 0.709


Selected on all students, then CV: 0.794
Selected inside each fold:         0.591


## 6. Mini project: a complete feature-engineering pipeline

In [ ]:
# Mini project, step 1: load and inspect the data
students = pd.read_csv("student_data_150.csv")
print(students.shape)
print(students.dtypes.to_string())
print("Missing values:", int(students.isna().sum().sum()), "| Class recorded as Unknown:", int((students["Class"] == "Unknown").sum()))
print("Duplicate students (ignoring the ID):", int(students.drop(columns="Student").duplicated().sum()))
print(students["Passed"].map({True: "Pass", False: "Fail"}).value_counts().to_string())

(150, 9)
Student              str
Class                str
Age              float64
StudyHours       float64
Attendance       float64
PreviousScore      int64
ExamScore        float64
Passed              bool
Source               str
Missing values: 0 | Class recorded as Unknown: 5
Duplicate students (ignoring the ID): 0
Passed
Pass    106
Fail     44


In [ ]:
# Steps 2 and 3: derived features (row-level, known at prediction time), then split before fitting anything
students["WeeklyStudyHours"] = students["StudyHours"] * 7
students["AttendanceCategory"] = pd.cut(students["Attendance"], bins=[0, 60, 80, 100], labels=["Low", "Medium", "High"],
                                        include_lowest=True).astype(str)
students["Class"] = students["Class"].replace("Unknown", np.nan)      # missing, for the imputer
features = ["StudyHours", "Attendance", "PreviousScore", "Age", "WeeklyStudyHours", "AttendanceCategory", "Class"]
X, y_all = students[features], students["Passed"].astype(int)           # 1 = pass, 0 = fail
X_train, X_test, y_train, y_test = train_test_split(X, y_all, test_size=0.15, random_state=42, stratify=y_all)
print(f"Train {len(X_train)} ({(y_train == 0).sum()} fail) | test {len(X_test)} ({(y_test == 0).sum()} fail): the test set waits")

Train 127 (37 fail) | test 23 (7 fail): the test set waits


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Steps 4 to 6: impute + scale numbers, impute + one-hot categories, SelectKBest, logistic regression: one pipeline
numeric_features = ["StudyHours", "Attendance", "PreviousScore", "Age", "WeeklyStudyHours"]
categorical_features = ["AttendanceCategory", "Class"]
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)])
model = Pipeline([("preprocessor", preprocessor),
                  ("feature_selection", SelectKBest(score_func=f_classif)),
                  ("classifier", LogisticRegression(max_iter=1000, random_state=42))])

# Step 7a: choose k by 5-fold CV on the training students only. We care about catching failing students,
# so we score F1 for the Fail class; everything (scaler, encoder, selector) is refitted inside each fold
fail_f1 = make_scorer(f1_score, pos_label=0)
search = GridSearchCV(model, {"feature_selection__k": [3, 5, 7, "all"]}, scoring=fail_f1, cv=cv).fit(X_train, y_train)
print(f"{len(search.best_estimator_.named_steps['preprocessor'].get_feature_names_out())} features after encoding")
for k, mean, std in zip(search.cv_results_["param_feature_selection__k"], search.cv_results_["mean_test_score"],
                        search.cv_results_["std_test_score"]):
    print(f"k={str(k):<4} CV F1 (fail) {mean:.3f} +/- {std:.3f}")
print("Best k:", search.best_params_["feature_selection__k"])

11 features after encoding
k=3    CV F1 (fail) 0.585 +/- 0.208
k=5    CV F1 (fail) 0.619 +/- 0.140
k=7    CV F1 (fail) 0.619 +/- 0.140
k=all  CV F1 (fail) 0.578 +/- 0.116
Best k: 5


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Step 7b: the final check, once, on the 23 test students
final = search.best_estimator_
predictions = final.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(classification_report(y_test, predictions, target_names=["Fail", "Pass"], digits=3))
print("Confusion matrix (rows actual, columns predicted; Fail, Pass):")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

Accuracy: 0.870
              precision    recall  f1-score   support

        Fail      0.833     0.714     0.769         7
        Pass      0.882     0.938     0.909        16

    accuracy                          0.870        23
   macro avg      0.858     0.826     0.839        23
weighted avg      0.867     0.870     0.867        23

Confusion matrix (rows actual, columns predicted; Fail, Pass):
[[ 5  2]
 [ 1 15]]


In [ ]:
# Step 8: which encoded features did SelectKBest keep, and with what F-scores (fitted on the 127 training students)?
names = final.named_steps["preprocessor"].get_feature_names_out()
selector = final.named_steps["feature_selection"]
report = pd.DataFrame({"Feature": names, "F-score": selector.scores_.round(2), "Kept": selector.get_support()})
print(report.sort_values("F-score", ascending=False).to_string(index=False))

                               Feature  F-score  Kept
                   numeric__StudyHours    40.88  True
             numeric__WeeklyStudyHours    40.88  True
                numeric__PreviousScore    20.68  True
                   numeric__Attendance    13.10  True
  categorical__AttendanceCategory_High    11.86  True
categorical__AttendanceCategory_Medium    11.44 False
                  categorical__Class_C     0.27 False
                  categorical__Class_B     0.26 False
                          numeric__Age     0.08 False
   categorical__AttendanceCategory_Low     0.03 False
                  categorical__Class_A     0.00 False
